## 05.02节练习参考答案

### 环境准备

In [ ]:
%pip install pypto==0.2.0 torch torch_npu numpy

In [ ]:
import os, sys
os.environ["TILE_FWK_DEVICE_ID"] = "0"

# 本 notebook 位于 answers/ 子目录下，src 包在其上一级目录。
# 把上层目录加入 sys.path，才能 from src.pto_layers import ...
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import pypto
import torch
import torch_npu
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import math

from src.pto_layers import PyPTOLinear, PyPTOReLU

本节的解答思路参考了 [《动手学深度学习》习题解答](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/ch05/ch05) ，并在此基础上补充了 PyPTO 的实现。  

### 练习5.2.1
查看初始化模块文档以了解不同的初始化方法。

**解答：**
通过查看深度学习框架文档，有以下初始化方法 （参考链接：https://pytorch.org/docs/stable/nn.init.html ）
* `torch.nn.init.uniform_(tensor, a=0.0, b=1.0)`：从均匀分布 *U(a,b)* 中提取填充输入张量。
* `torch.nn.init.normal_(tensor, mean=0.0, std=1.0)`：从正态分布 *N(mean,std²)* 中提取填充输入张量。
* `torch.nn.init.constant_(tensor, val)`：以一确定数值初始化张量。
* `torch.nn.init.ones_(tensor)`：用标量值 1 填充输入张量。
* `torch.nn.init.zeros_(tensor)`：用标量值 0 填充输入张量。
* `torch.nn.init.eye_(tensor)`：用单位矩阵填充二维输入张量
* `torch.nn.init.xavier_uniform_(tensor, gain=1.0)`：从均匀分布 *U(−a,a)* 中采样，初始化输入张量，其中 a 的值由如下公式确定
$$
a = gain * \sqrt{\frac{6}{fan_{in} + fan_{out}}}
$$
其中 gain 的取值如下表所示
| 非线性函数       | gain 值                          |
|-----------------|---------------------------------|
| Linear/Identity |                                 |
| Conv1D          | $1$                             |
| Conv2D          | $1$                             |
| Conv3D          | $1$                             |
| Sigmoid         | $1$                             |
| Tanh            | $\dfrac{5}{3}$                  |
| ReLU            | $\sqrt{2}$                      |
| Leaky ReLU      | $\sqrt{\dfrac{2}{1 + negative\_slope^2}}$ |
| SELU            | $1$ (adaptive)                  |
* `torch.nn.init.xavier_normal_(tensor, gain=1.0)`:从正态分布 *N(0,std²)* 中采样，初始化输入张量，其中 std 值由下式确定：
$$
a = gain * \sqrt{\frac{2}{fan_{in} + fan_{out}}}
$$
* `torch.nn.init.kaiming_uniform_(tensor, a=0, mode='fan_in', nonlinearity='leaky_relu')`:服从均匀分布 *U(−bound,bound)*，其中 bound 值由下式确定
$$
bound = gain \cdot \sqrt{\dfrac{3}{fan_{mode}}}
$$
* `torch.nn.init.kaiming_normal_(tensor, a=0, mode='fan_in', nonlinearity='leaky_relu')`:服从从正态分布 *N(0,std 2)* 中采样，其中 std 值由下式确定
$$
std = \frac{gain}{\sqrt{fan_{mode}}}
$$
* `torch.nn.init.trunc_normal_(tensor, mean=0.0, std=1.0, a=- 2.0, b=2.0)`:用从截断的正态分布中提取的值填充输入张量。这些值实际上是从正态分布 *N(mean,std²)* 中提取的。
* `torch.nn.init.sparse_(tensor, sparsity, std=0.01)`：将 2D 输入张量填充为稀疏矩阵，其中非零元素将从正态分布 *N(0,0.01)* 中提取。

### 练习5.2.2
构建包含共享参数层的多层感知机并对其进行训练。在训练过程中，观察模型各层的参数和梯度。

**解答：**
在训练过程中，我们每个 epoch 都打印了每层的参数和梯度。可以看到 shared_fc 层的参数和梯度都是相同的，因为它们共享同一个参数。

In [4]:
# 模型参数
input_size = 4
hidden_size = 8
output_size = 4
lr = 0.01
epochs = 2

# 构建带有共享参数层的多层感知机
shared_fc = nn.Linear(hidden_size, hidden_size)
MLP = nn.Sequential(nn.Linear(input_size, hidden_size), nn.ReLU(),
                    shared_fc, nn.ReLU(),
                    shared_fc, nn.ReLU(),
                    nn.Linear(hidden_size, output_size)
)

# 训练数据
X = torch.randn(1, input_size)
Y = torch.randn(1, output_size)
# 优化器
optimizer = optim.SGD(MLP.parameters(), lr=lr)
# 训练模型
for epoch in range(epochs):
    # 前向传播和计算损失
    Y_pred = MLP(X)
    loss = nn.functional.mse_loss(Y_pred, Y)
    # 反向传播和更新梯度
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # 打印每层的参数和梯度
    for name, param in MLP.named_parameters():
        print(name, param.data, param.grad)
    print('Epoch: {}, Loss: {}'.format(epoch, loss.item()))

0.weight tensor([[-0.0436, -0.0382,  0.0281, -0.4413],
        [ 0.1321, -0.1262, -0.0535, -0.3650],
        [ 0.0311,  0.4139, -0.2118, -0.4179],
        [ 0.0028, -0.0102,  0.2418,  0.4724],
        [-0.1093, -0.2947,  0.4902,  0.3650],
        [ 0.4527,  0.3789, -0.0423, -0.3782],
        [ 0.1777, -0.2236, -0.0522,  0.3291],
        [-0.1260,  0.2235, -0.4281,  0.2024]]) tensor([[ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0060,  0.0030, -0.0705,  0.0169],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0005,  0.0002, -0.0057,  0.0014]])
0.bias tensor([-0.3935, -0.0067,  0.4752, -0.4394, -0.2303, -0.0991, -0.4094,  0.1758]) tensor([0.0000, 0.0000, 0.0475, 0.0000, 0.0000, 0.0000, 0.0000, 0.0039])
2.weight tensor([[-0.2743,  0.3346,  0.3292, -0.0024, -0.1955,  0.0399,  0.1813,  0.2722],
 

**PyPTO 版**

In [6]:
# 模型参数
input_size = 4
hidden_size = 8
output_size = 4
lr = 0.01
epochs = 2

# 构建带有共享参数层的多层感知机
shared_fc = PyPTOLinear(hidden_size, hidden_size)
MLP = nn.Sequential(PyPTOLinear(input_size, hidden_size), PyPTOReLU(),
                    shared_fc, PyPTOReLU(),
                    shared_fc, PyPTOReLU(),
                    PyPTOLinear(hidden_size, output_size)
)

# 训练数据
X = torch.randn(1, input_size).npu()
Y = torch.randn(1, output_size).npu()
# 优化器
optimizer = optim.SGD(MLP.parameters(), lr=lr)
# 训练模型
for epoch in range(epochs):
    # 前向传播和计算损失
    Y_pred = MLP(X)
    loss = nn.functional.mse_loss(Y_pred, Y)
    # 反向传播和更新梯度
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # 打印每层的参数和梯度
    for name, param in MLP.named_parameters():
        print(name, param.data, param.grad)
    print('Epoch: {}, Loss: {}'.format(epoch, loss.item()))

0.weight tensor([[-0.0823, -0.3106,  0.2430,  0.2303],
        [ 0.0129,  0.1242,  0.1136, -0.2268],
        [ 0.4487,  0.3730, -0.0951, -0.1281],
        [ 0.0928, -0.3642,  0.0573,  0.3196],
        [-0.3204,  0.3209, -0.0197, -0.3733],
        [-0.2287,  0.0817, -0.1699,  0.1254],
        [ 0.2693, -0.1000,  0.3790,  0.0194],
        [ 0.3520,  0.2423, -0.2559, -0.1212]], device='npu:0') tensor([[-0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0136, -0.0193, -0.0248, -0.0363],
        [-0.0019,  0.0027,  0.0035,  0.0051],
        [ 0.0052, -0.0075, -0.0096, -0.0140],
        [-0.0002,  0.0003,  0.0004,  0.0006],
        [ 0.0000, -0.0000, -0.0000, -0.0000],
        [-0.0025,  0.0036,  0.0046,  0.0067],
        [ 0.0097, -0.0137, -0.0176, -0.0258]], device='npu:0')
0.bias tensor([-0.3870,  0.3468, -0.0100,  0.3391,  0.1543, -0.2095,  0.3756,  0.4632],
       device='npu:0') tensor([ 0.0000,  0.0283, -0.0040,  0.0109, -0.0004,  0.0000, -0.0053,  0.0201],
       device='npu:0')
2.weig

### 练习5.2.3
为什么共享参数是个好主意？

**解答：**
1. 节约内存：共享参数可以减少模型中需要存储的参数数量，从而减少内存占用。

2. 加速收敛：共享参数可以让模型更加稳定，加速收敛。

3. 提高泛化能力：共享参数可以帮助模型更好地捕捉数据中的共性，提高模型的泛化能力。

4. 加强模型的可解释性：共享参数可以让模型更加简洁明了，加强模型的可解释性。